# Koduppgift 5

### koduppgift 8

här importerar vi våra bibliotek och skapar en X som innehåller 3 columns och 1000 rows. Vi använder random för att få ut random värde i dom 

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

X = np.random.rand(1000,3)
print(X[0:5])


[[0.13002309 0.25915087 0.72767703]
 [0.93124893 0.85001313 0.88625024]
 [0.93584324 0.65363133 0.22696599]
 [0.42110189 0.26652199 0.30579242]
 [0.76674879 0.10270996 0.55296285]]


Här använder vi PCA med n_components för att reducera ner det till 2 columns istället för 3

In [11]:
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)
print(X2D[0:5])

[[-0.27753719  0.36269415]
 [ 0.66549892  0.13275264]
 [ 0.25910592 -0.39784533]
 [-0.27833349 -0.08283572]
 [-0.09682919  0.13170106]]


Här försöker vi återskapa våran 3 columns set men får false då den inte lyckas då skillnaden är förstår för att kunna återskapa.

In [12]:
X3d_inv = pca.inverse_transform(X2D)
print(np.allclose(X3d_inv,X))

False


### koduppgift 9

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

In [2]:
df = pd.read_csv('./dataset/car_price_dataset.csv', sep=';')

df.head(5)

,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


In [ ]:
X = df.drop(['Price', 'Doors'], axis=1)
y = df['Price']

categorical_cols = ['Brand', 'Model', 'Fuel_Type', 'Transmission']
numerical_cols = ['Year', 'Engine_Size', 'Mileage', 'Owner_Count']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ],
    remainder='passthrough'
)

X_prepared = preprocessor.fit_transform(X)

pca = PCA(n_components=0.7)
X_pca = pca.fit_transform(X_prepared)

print(f"Antal dimensioner efter PCA: {pca.n_components_}")

Antal dimensioner efter PCA: 8


In [4]:
X_full, X_test, y_full, y_test = train_test_split(X_pca, y, train_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_full, y_full, test_size=0.25, random_state=42)

In [5]:
rfr = RandomForestRegressor(random_state=42)

hyperparams = {
    'n_estimators': [50, 100, 150, 200, 300],
    'max_depth': [2, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

rfr_gs = GridSearchCV(
    estimator=rfr,
    param_grid=hyperparams,
    scoring='neg_mean_squared_error')

rfr_gs.fit(X_train, y_train)

rfr_pred = rfr_gs.predict(X_val)

rmse_rfr = root_mean_squared_error(y_val, rfr_pred)

print('best params: ', rfr_gs.best_params_)
print('rmse test: ', rmse_rfr)

best params:  {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 300}
rmse test:  652.9747205096766


In [6]:
rfr_gs_2 =GridSearchCV(
    estimator=rfr,
    param_grid=hyperparams,
    scoring='neg_mean_squared_error')


rfr_gs_2.fit(X_full, y_full)

rfr_pred_2 = rfr_gs.predict(X_test)

rmse_rfr_2 = root_mean_squared_error(y_test, rfr_pred_2)

print('best params: ', rfr_gs_2.best_params_)
print('rmse test: ', rmse_rfr_2)

best params:  {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 200}
rmse test:  666.2927616787817


#### Gemförelse mellan detta och [koduppgiften 15 i kapitel 3](koduppgift3.ipynb)

efter att ha gjort en pca så presterar modellen sämre på datan som vi har. Jag antar att jag hade kunnat fin kalibrera detta för att den skulle blir bättre men jag kände att det kunde vara kul att bara skriva ut koden och se vad som hände. 

i slut resultat så blir modellen ca 39 rmse sämre än den vanliga modellen utan pca.